# Assignment 5 · Task 2 — Parts B & C: NST & Video Pipeline

This notebook runs the full pipeline for Neural Style Transfer, Ablations, and Video Compositing using the GPU.

**Outputs Generated:**
- `grid.png`: NST sanity check grid
- `beta_alpha_ablation.png`: Style weight sweep
- `layer_ablation.png`: Shallow vs Deep layers
- `matting_overlay.png`: Matting model performance on video frames
- `feature_maps.png`: VGG19 internal activations
- `branded_poster.png`: 1024x1024 branded still
- `stylized_background.mp4`: Background-only stylization
- `stylized_subject.mp4`: Subject-only stylization
- `stylized_full.mp4`: Full frame stylization

> **Prerequisite:** You should have trained the matting model (Part A) and have `matting_weights.pth` available, or use the provided weights.

## 1 · Environment Check

In [ ]:
import subprocess, sys, os, shutil, re
from pathlib import Path

print('=' * 60)
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
!df -h /kaggle/working

## 2 · Clone GitHub Repo
> **Edit `GITHUB_REPO`** to your actual repository URL before running.

In [ ]:
# ✏️  CHANGE THIS
GITHUB_REPO = 'https://github.com/YOUR_USERNAME/CV-AS-5-TASK2'

REPO_DIR = Path('/kaggle/working/repo')
if REPO_DIR.exists():
    print('Repo present — pulling latest ...')
    !git -C {REPO_DIR} pull
else:
    !git clone --depth 1 {GITHUB_REPO} {REPO_DIR}

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print('\n[OK] Repo ready.')

## 3 · Install Dependencies

In [ ]:
req = REPO_DIR / 'requirements.txt'
if req.exists():
    !pip install -q -r {req}
else:
    !pip install -q tqdm pyyaml Pillow opencv-python matplotlib
print('[OK] Dependencies ready.')

## 4 · Prepare Input Files

In [ ]:
import requests

# 1. Handle Weights
weights_dest = REPO_DIR / 'matting_weights.pth'
if not weights_dest.exists():
    # Check if they exist in /kaggle/working/ (from a Part A run)
    source = Path('/kaggle/working/matting_weights.pth')
    if source.exists():
        shutil.copy(source, weights_dest)
        print('Found weights in /kaggle/working/')
    else:
        print('WARNING: matting_weights.pth not found. Please upload it to /kaggle/working/repo/')

# 2. Handle Input Video
video_path = REPO_DIR / 'input_video.mp4'
if not video_path.exists():
    print('Downloading sample video...')
    # Using a public sample video (e.g., from Pexels or similar)
    sample_url = 'https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/person-bicycle-car-detection.mp4'
    try:
        r = requests.get(sample_url, stream=True)
        with open(video_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024):
                if chunk: f.write(chunk)
        print('Sample video downloaded.')
    except Exception as e:
        print(f'Failed to download sample video: {e}')
        print('Please upload input_video.mp4 manually.')

print(f'Weights exists: {weights_dest.exists()}')
print(f'Video exists  : {video_path.exists()}')

## 5 · Run Full Pipeline

In [ ]:
os.chdir(REPO_DIR)
!python run_all_outputs.py

## 6 · Display Results

In [ ]:
from IPython.display import display, Image as IPyImage
output_dir = REPO_DIR / 'outputs'

images = ['grid.png', 'beta_alpha_ablation.png', 'layer_ablation.png', 'matting_overlay.png', 'feature_maps.png', 'branded_poster.png']
for img_name in images:
    p = output_dir / img_name
    if p.exists():
        print(f'--- {img_name} ---')
        display(IPyImage(str(p)))
    else:
        print(f'MISSING: {img_name}')

## 7 · Package Outputs

In [ ]:
zip_base = '/kaggle/working/parts_bc_results'
shutil.make_archive(zip_base, 'zip', output_dir)
print(f'Archive created: {zip_base}.zip')
!ls -lh {zip_base}.zip